In [ ]:
!pip install torch torchvision timm scikit-learn albumentations optuna

import os, glob, random
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 13.2 MB/s eta 0:00:00


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
# Path to dataset
data_dir = "/content/drive/MyDrive/enhanced_clahe"
labels_df = pd.read_csv("/content/drive/MyDrive/data.csv")

# Clean missing
labels_df = labels_df.dropna(subset=["superclass", "subclass", "image_id"]).reset_index(drop=True)

# Folder name = superclass_subclass
labels_df["folder"] = labels_df["superclass"] + "_" + labels_df["subclass"]

# Encode labels
le = LabelEncoder()
labels_df["label"] = le.fit_transform(labels_df["subclass"])
print("Classes:", le.classes_)

# Split dataset
train_df, temp_df = train_test_split(labels_df, test_size=0.3, stratify=labels_df["label"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


Classes: ['bd' 'md' 'pd']
Train: 378, Val: 81, Test: 81


In [ ]:
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.GaussianBlur(img, (5,5), 0)
    img = cv2.resize(img, (224,224))
    return img

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9,1.1), translate_percent=(0.1,0.1),
             rotate=(-15,15), shear=(-5,5), p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406),
                std=(0.229,0.224,0.225)),
    ToTensorV2()
])


In [ ]:
def tile_image_for_mil(img_rgb, patch_size=224, stride=224, max_patches=64, mode="grid"):
    H, W = img_rgb.shape[:2]
    patches = []
    for y in range(0, max(1, H - patch_size + 1), stride):
        for x in range(0, max(1, W - patch_size + 1), stride):
            crop = img_rgb[y:y+patch_size, x:x+patch_size]
            if crop.shape[:2] == (patch_size, patch_size):
                patches.append(crop)
    if len(patches) > max_patches:
        idx = np.linspace(0, len(patches)-1, max_patches).astype(int)
        patches = [patches[i] for i in idx]
    if len(patches) == 0:
        patches = [cv2.resize(img_rgb, (patch_size, patch_size))]
    return patches

class MILBagDataset(Dataset):
    def __init__(self, img_root, df, augment, patch_size=224, stride=224, max_patches=32):
        self.img_root, self.df, self.augment = img_root, df.reset_index(drop=True), augment
        self.patch_size, self.stride, self.max_patches = patch_size, stride, max_patches

    def _resolve_img_path(self, row):

        folder = row["folder"]
        image_id = str(row["image_id"])

    # Multiple possible patterns
        patterns = [
        f"{folder}_20x_{image_id}_clahe*",
        f"{folder}_40x_{image_id}_clahe*",
        f"{folder}_20x_{image_id}*",
        f"{folder}_40x_{image_id}*",
        f"{folder}_{image_id}*",
        f"{image_id}*"
    ]

        for patt in patterns:
          files = glob.glob(os.path.join(self.img_root, folder, patt))
          if files:
            return files[0]

    # If nothing found, return None instead of crashing
        return None

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
      row = self.df.iloc[idx]
      img_path = self._resolve_img_path(row)
      if img_path is None:
        raise FileNotFoundError(f"No file found for row: {row.to_dict()}")

      raw = cv2.imread(img_path)
      raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
      raw = cv2.GaussianBlur(raw, (5,5), 0)

      patches = tile_image_for_mil(raw, patch_size=self.patch_size, stride=self.stride,
                                 max_patches=self.max_patches)
      tensor_patches = [self.augment(image=p)["image"] for p in patches]
      bag_tensor = torch.stack(tensor_patches, dim=0)
      return bag_tensor, int(row["label"]), row["image_id"]


def mil_collate(batch):
    bags, labels, ids = zip(*batch)
    return list(bags), torch.tensor(labels, dtype=torch.long), list(ids)


In [ ]:
class MILModel(nn.Module):
    def __init__(self, backbone, num_classes, agg="mean"):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.num_features, num_classes)
        self.agg = agg

    def forward(self, bag):
        if bag.dim() == 4:
            bag = bag.unsqueeze(0)
        B, n, C, H, W = bag.shape
        bag = bag.view(B*n, C, H, W)

        feats = self.backbone(bag)
        logits = self.classifier(feats)
        logits = logits.view(B, n, -1)

        if self.agg == "mean":
            bag_logits = logits.mean(dim=1)
        elif self.agg == "max":
            bag_logits = logits.max(dim=1).values
        elif self.agg == "lse":
            bag_logits = torch.logsumexp(logits, dim=1)
        else:
            bag_logits = logits.mean(dim=1)
        return bag_logits, logits


In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch_mil(model, loader, optimizer=None, train=True, pooling="mean"):
    if train: model.train()
    else: model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    for bags, labels, ids in loader:
        bag = bags[0].to(device)
        labels = labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(device.type=="cuda")):
                inst_logits = model(bag)[0]  # bag_logits
                loss = criterion(inst_logits, labels)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                inst_logits = model(bag)[0]
                loss = criterion(inst_logits, labels)

        total_loss += float(loss.item())
        preds = inst_logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, prec, rec, f1


In [ ]:
import optuna
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    bag_size = trial.suggest_categorical("bag_size", [8, 16, 32])
    pooling = trial.suggest_categorical("pooling", ["mean", "max", "lse"])
    epochs = 5

    # datasets
    train_dataset = MILBagDataset(data_dir, train_df, augment, max_patches=bag_size)
    val_dataset   = MILBagDataset(data_dir, val_df,   augment, max_patches=bag_size)

    # filter missing files (skip rows with no file)
    train_dataset.df = train_dataset.df[train_dataset.df.apply(lambda r: train_dataset._resolve_img_path(r) is not None, axis=1)]
    val_dataset.df   = val_dataset.df[val_dataset.df.apply(lambda r: val_dataset._resolve_img_path(r) is not None, axis=1)]

    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=mil_collate)
    val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False, collate_fn=mil_collate)

    # model
    backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
    model = MILModel(backbone, num_classes=len(le.classes_), agg=pooling).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = 0
    for _ in range(epochs):
        run_epoch_mil(model, train_loader, optimizer=optimizer, train=True, pooling=pooling)
        vl_loss, vl_acc, _, _, _ = run_epoch_mil(model, val_loader, train=False, pooling=pooling)
        best_val_acc = max(best_val_acc, vl_acc)

    return best_val_acc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # try 10 sets

print("Best hyperparameters:", study.best_params)
print("Best Val Acc:", study.best_value)


[I 2025-10-03 07:20:22,542] A new study created in memory with name: no-name-d0bf82de-46e4-4e0c-8e88-cb90eb4aab82
[I 2025-10-03 07:25:52,567] Trial 0 finished with value: 0.37037037037037035 and parameters: {'lr': 0.0007788377866485387, 'weight_decay': 0.0005002265488391214, 'bag_size': 8, 'pooling': 'mean'}. Best is trial 0 with value: 0.37037037037037035.
[I 2025-10-03 07:31:16,303] Trial 1 finished with value: 0.5308641975308642 and parameters: {'lr': 0.00011864153750186548, 'weight_decay': 0.00034519043833741795, 'bag_size': 8, 'pooling': 'max'}. Best is trial 1 with value: 0.5308641975308642.
[I 2025-10-03 07:41:55,002] Trial 2 finished with value: 0.37037037037037035 and parameters: {'lr': 0.0002032137642701907, 'weight_decay': 0.0011133244220729087, 'bag_size': 32, 'pooling': 'max'}. Best is trial 1 with value: 0.5308641975308642.
[I 2025-10-03 07:48:47,599] Trial 3 finished with value: 0.7530864197530864 and parameters: {'lr': 1.0657739549379864e-05, 'weight_decay': 0.000188855

Best hyperparameters: {'lr': 1.0657739549379864e-05, 'weight_decay': 0.00018885518469095871, 'bag_size': 16, 'pooling': 'lse'}
Best Val Acc: 0.7530864197530864


In [ ]:
best_params = study.best_params

train_dataset = MILBagDataset(data_dir, train_df, augment, max_patches=best_params["bag_size"])
val_dataset   = MILBagDataset(data_dir, val_df,   augment, max_patches=best_params["bag_size"])
test_dataset  = MILBagDataset(data_dir, test_df,  augment, max_patches=best_params["bag_size"])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,  collate_fn=mil_collate)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False, collate_fn=mil_collate)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False, collate_fn=mil_collate)

backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
model = MILModel(backbone, num_classes=len(le.classes_), agg=best_params["pooling"]).to(device)

optimizer = optim.AdamW(model.parameters(),
                        lr=best_params["lr"],
                        weight_decay=best_params["weight_decay"])

best_val_acc, stale, patience = 0.0, 0, 5
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_prec, tr_rec, tr_f1 = run_epoch_mil(model, train_loader, optimizer, train=True, pooling=best_params["pooling"])
    vl_loss, vl_acc, vl_prec, vl_rec, vl_f1 = run_epoch_mil(model, val_loader, train=False, pooling=best_params["pooling"])

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Acc {tr_acc:.4f} | Val Acc {vl_acc:.4f} | F1 {vl_f1:.4f}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        stale = 0
        torch.save(model.state_dict(), "swin_tiny_best_tuned.pt")
    else:
        stale += 1
        if stale >= patience:
            print("⏹️ Early stopping.")
            break

print("✅ Best Val Acc after tuning:", best_val_acc)


Epoch 01/20 | Train Acc 0.4921 | Val Acc 0.4938 | F1 0.4618
Epoch 02/20 | Train Acc 0.6614 | Val Acc 0.6296 | F1 0.5734
Epoch 03/20 | Train Acc 0.7275 | Val Acc 0.6173 | F1 0.6043
Epoch 04/20 | Train Acc 0.7434 | Val Acc 0.7901 | F1 0.7977
Epoch 05/20 | Train Acc 0.7937 | Val Acc 0.8395 | F1 0.8393
Epoch 06/20 | Train Acc 0.8360 | Val Acc 0.7901 | F1 0.7783
Epoch 07/20 | Train Acc 0.8439 | Val Acc 0.8025 | F1 0.8083
Epoch 08/20 | Train Acc 0.8439 | Val Acc 0.8148 | F1 0.8091
Epoch 09/20 | Train Acc 0.9048 | Val Acc 0.8519 | F1 0.8516
Epoch 10/20 | Train Acc 0.9127 | Val Acc 0.8642 | F1 0.8643
Epoch 11/20 | Train Acc 0.9312 | Val Acc 0.8889 | F1 0.8871
Epoch 12/20 | Train Acc 0.9365 | Val Acc 0.7037 | F1 0.6952
Epoch 13/20 | Train Acc 0.9550 | Val Acc 0.8642 | F1 0.8625
Epoch 14/20 | Train Acc 0.9788 | Val Acc 0.9136 | F1 0.9147
Epoch 15/20 | Train Acc 0.9444 | Val Acc 0.8148 | F1 0.8147
Epoch 16/20 | Train Acc 0.9127 | Val Acc 0.8642 | F1 0.8620
Epoch 17/20 | Train Acc 0.9815 | Val Acc

In [ ]:
# ======================================
# 🔹 CNN + Swin Hybrid MIL Model
# ======================================
class CNN_SwinMIL(nn.Module):
    def __init__(self, num_classes, agg="mean"):
        super().__init__()
        # CNN for local features
        self.cnn = timm.create_model("resnet18", pretrained=True, num_classes=0, global_pool="avg")
        # Swin Transformer for global features
        self.swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)

        self.fc = nn.Linear(self.cnn.num_features + self.swin.num_features, num_classes)
        self.agg = agg

    def forward(self, bag):
        if bag.dim() == 4:   # single bag case
            bag = bag.unsqueeze(0)
        B, n, C, H, W = bag.shape
        bag = bag.view(B*n, C, H, W)   # flatten patches

        # Extract features
        cnn_feat = self.cnn(bag)   # [B*n, cnn_dim]
        swin_feat = self.swin(bag) # [B*n, swin_dim]

        # Fuse features
        fused = torch.cat([cnn_feat, swin_feat], dim=1)  # [B*n, total_dim]

        # Class logits per patch
        logits = self.fc(fused).view(B, n, -1)

        # MIL aggregation
        if self.agg == "mean":
            bag_logits = logits.mean(dim=1)
        elif self.agg == "max":
            bag_logits = logits.max(dim=1).values
        elif self.agg == "lse":
            bag_logits = torch.logsumexp(logits, dim=1)
        else:
            bag_logits = logits.mean(dim=1)

        return bag_logits, logits


# ======================================
# 🔹 Training Loop (same as before)
# ======================================
criterion = nn.CrossEntropyLoss()

def run_epoch_mil(model, loader, optimizer=None, train=True):
    if train: model.train()
    else: model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    for bags, labels, ids in loader:
        bag = bags[0].to(device)
        labels = labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(device.type=="cuda")):
                bag_logits, _ = model(bag)
                loss = criterion(bag_logits, labels)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                bag_logits, _ = model(bag)
                loss = criterion(bag_logits, labels)

        total_loss += float(loss.item())
        preds = bag_logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, prec, rec, f1


# ======================================
# 🔹 Initialize + Train
# ======================================
model = CNN_SwinMIL(num_classes=len(le.classes_), agg="mean").to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

best_val_acc, stale, patience = 0.0, 0, 5
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_prec, tr_rec, tr_f1 = run_epoch_mil(model, train_loader, optimizer, train=True)
    vl_loss, vl_acc, vl_prec, vl_rec, vl_f1 = run_epoch_mil(model, val_loader, train=False)

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Acc {tr_acc:.4f} | Val Acc {vl_acc:.4f} | F1 {vl_f1:.4f}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        stale = 0
        torch.save(model.state_dict(), "cnn_swin_mil_best.pt")
    else:
        stale += 1
        if stale >= patience:
            print("⏹️ Early stopping.")
            break

print("✅ Best Val Acc with CNN+Swin hybrid:", best_val_acc)


Epoch 01/20 | Train Acc 0.3757 | Val Acc 0.3457 | F1 0.1776
Epoch 02/20 | Train Acc 0.3889 | Val Acc 0.3704 | F1 0.2002
Epoch 03/20 | Train Acc 0.4206 | Val Acc 0.3457 | F1 0.1809
Epoch 04/20 | Train Acc 0.5026 | Val Acc 0.3457 | F1 0.1976
Epoch 05/20 | Train Acc 0.5212 | Val Acc 0.3704 | F1 0.2272
Epoch 06/20 | Train Acc 0.5661 | Val Acc 0.4444 | F1 0.3779
Epoch 07/20 | Train Acc 0.6534 | Val Acc 0.3457 | F1 0.1988
Epoch 08/20 | Train Acc 0.6429 | Val Acc 0.3580 | F1 0.2730
Epoch 09/20 | Train Acc 0.6746 | Val Acc 0.3580 | F1 0.2087
Epoch 10/20 | Train Acc 0.7196 | Val Acc 0.3210 | F1 0.1967
Epoch 11/20 | Train Acc 0.7249 | Val Acc 0.3580 | F1 0.2502
⏹️ Early stopping.
✅ Best Val Acc with CNN+Swin hybrid: 0.4444444444444444
